# Demo 1 - Lake exploration

**Workshop:** F16 Advanced analytics (notebooks / ML) - Microsoft Sentinel data lake
**Pool:** Small · **Time:** ~25 min

Goal: get comfortable reading lake-tier tables into Spark DataFrames and shaping them.

> Set `WORKSPACE` below to your workspace name (from `list_databases()`), then run the cells
> top to bottom. The first run starts a Spark session (3-6 min) - that is normal.

## 1. Connect to the lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your data lake. The
`spark` session is created for you by the Microsoft Sentinel kernel, so never build your
own - the provider needs the one the kernel gave you.

`list_databases()` returns the workspaces this lake exposes. You will see the Log Analytics
workspaces you have access to, plus `System Tables`, which is where custom tables written
by notebooks land.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider

# `spark` is provided by the Microsoft Sentinel kernel - do not create your own SparkSession.
data_provider = MicrosoftSentinelProvider(spark)

# Which workspaces (databases) exist in this lake?
print(data_provider.list_databases())

## 1b. Preflight - what's on this pool?

Run this **once** before a demo. The later notebooks use seaborn, scikit-learn, statsmodels,
networkx and geopy.

**There is no fix if something is missing.** The data lake runtime supports only the
[Azure Synapse Spark 3.4 libraries](https://github.com/microsoft/synapse-spark-runtime/tree/main#readme)
plus the Sentinel provider - `%pip install` and custom libraries are not supported. So this
cell tells you which demos to drop from the running order, not what to install.

Demo 7 is the only one that degrades gracefully: it falls back to a built-in haversine
calculation when `geopy` is absent.

In [ ]:
import importlib

REQUIRED = ["pandas", "numpy", "matplotlib"]                       # demos 1-18
OPTIONAL = ["seaborn", "sklearn", "statsmodels", "networkx", "scipy", "geopy"]

for mod in REQUIRED + OPTIONAL:
    try:
        m = importlib.import_module(mod)
        print(f"  OK       {mod:12} {getattr(m, '__version__', '')}")
    except ImportError:
        tag = "REQUIRED" if mod in REQUIRED else "optional"
        print(f"  MISSING  {mod:12} ({tag}) - cannot be installed; see note below")

## 2. Set your workspace and list its tables

Pick one of the workspace names printed above and put it in `WORKSPACE`. Every later cell
reads from it.

`list_tables()` returns `TableInfo` objects rather than plain strings, which is why we pull
`.name` off each one before sorting. Sorting the objects directly raises `TypeError` because
`TableInfo` defines no ordering. The Microsoft class reference still documents this as
returning `list[str]`, so trust the runtime, not the docs.

In [ ]:
WORKSPACE = "your-workspace-name"   # <-- replace with a name printed above

tables = data_provider.list_tables(WORKSPACE)
print(f"{len(tables)} tables in {WORKSPACE}:")

# list_tables returns TableInfo objects, not plain strings (the class reference still
# documents list[str]) and TableInfo defines no ordering - so sort on the name.
table_names = sorted(str(getattr(t, "name", t)) for t in tables)
for name in table_names:
    print(" -", name)

## 3. Read a table

`read_table` hands back a **lazy** Spark DataFrame. Nothing is read yet - Spark has only
recorded what you asked for.

Work happens when you call an **action**: `.show()`, `.count()`, `.toPandas()`. Everything
before that (`select`, `filter`, `groupBy`) is a **transformation**, which just adds to the
plan. This is why you can chain a dozen operations over years of data and it costs nothing
until the last line.

`printSchema()` is effectively free. It reads the table's metadata, not its rows.

In [ ]:
df = data_provider.read_table("SigninLogs", WORKSPACE)

# Inspect the schema (cheap - no data scan)
df.printSchema()

## 4. Project a few columns and preview

`select` narrows to the columns you care about, and because the lake is column-oriented,
Spark only reads those columns off disk. Selecting five columns out of sixty is roughly a
twelvefold saving before you have filtered a single row.

`.show(20)` is the action that finally triggers the read.

In [ ]:
from pyspark.sql.functions import col

(df.select("TimeGenerated", "UserPrincipalName", "IPAddress", "AppDisplayName", "ResultType")
   .orderBy(col("TimeGenerated").desc())
   .show(20, truncate=False))

## 5. Aggregate - sign-ins per user

`groupBy` plus `agg` is the workhorse: collapse many rows into one row per group.

This is also the habit that keeps notebooks fast. Aggregate in Spark, where the work is
distributed across the pool, and only bring back the small result. Pulling raw rows to the
driver and counting them in pandas is the classic way to make a notebook hang.

In [ ]:
from pyspark.sql.functions import count, desc

signins_per_user = (df.groupBy("UserPrincipalName")
                      .agg(count("*").alias("SigninCount"))
                      .orderBy(desc("SigninCount")))

signins_per_user.show(20, truncate=False)

## 6. Filter + order - busiest source IPs

Same pattern, two columns at once: `count("*")` for total events and `countDistinct` for
how many different users came from each IP.

That second number is the interesting one. One IP with many events is usually a proxy or a
VPN concentrator. One IP with many *distinct users* and a high failure rate is a rather
different story.

Note the `filter` comes after the `agg` - it is filtering the aggregated result, not the
raw rows.

In [ ]:
from pyspark.sql.functions import countDistinct

top_ips = (df.groupBy("IPAddress")
             .agg(count("*").alias("Events"),
                  countDistinct("UserPrincipalName").alias("DistinctUsers"))
             .filter(col("Events") > 50)
             .orderBy(desc("Events")))

top_ips.show(20, truncate=False)

## Recap

- `read_table(table, workspace)` -> lazy Spark DataFrame.
- Transformations (`select`/`filter`/`groupBy`) are lazy; actions (`show`/`count`/`toPandas`)
  execute.
- Aggregate **before** display (the VS Code grid caps at 100,000 rows).

**Stretch:** pick another table from cell 2 and repeat cells 3-6 against it.